In [1]:
import pandas as pd
import numpy as np
import pickle
import os
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score

data_2017_path = r"C:\Users\lenovo\Desktop\Projects\Network-IDS\data"
data_2018_path = r"C:\Users\lenovo\Desktop\Projects\Network-IDS\data\cicids2018"
models_path = r"C:\Users\lenovo\Desktop\Projects\Network-IDS\models"

print("Imports done")

Imports done


In [2]:
csv_files_2018 = [f for f in os.listdir(data_2018_path) if f.endswith('.csv')]
print(f"Found {len(csv_files_2018)} files:")
for f in csv_files_2018:
    print(f"  {f}")

# Load first file, just a sample to inspect structure
sample_path = os.path.join(data_2018_path, csv_files_2018[0])
df_2018_sample = pd.read_csv(sample_path, low_memory=False, nrows=5000)
df_2018_sample.columns = df_2018_sample.columns.str.strip()

print(f"\nShape: {df_2018_sample.shape}")
print(f"\nColumns:")
for col in df_2018_sample.columns:
    print(f"  '{col}'")

Found 3 files:
  Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv
  Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv
  Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv

Shape: (5000, 80)

Columns:
  'Dst Port'
  'Protocol'
  'Timestamp'
  'Flow Duration'
  'Tot Fwd Pkts'
  'Tot Bwd Pkts'
  'TotLen Fwd Pkts'
  'TotLen Bwd Pkts'
  'Fwd Pkt Len Max'
  'Fwd Pkt Len Min'
  'Fwd Pkt Len Mean'
  'Fwd Pkt Len Std'
  'Bwd Pkt Len Max'
  'Bwd Pkt Len Min'
  'Bwd Pkt Len Mean'
  'Bwd Pkt Len Std'
  'Flow Byts/s'
  'Flow Pkts/s'
  'Flow IAT Mean'
  'Flow IAT Std'
  'Flow IAT Max'
  'Flow IAT Min'
  'Fwd IAT Tot'
  'Fwd IAT Mean'
  'Fwd IAT Std'
  'Fwd IAT Max'
  'Fwd IAT Min'
  'Bwd IAT Tot'
  'Bwd IAT Mean'
  'Bwd IAT Std'
  'Bwd IAT Max'
  'Bwd IAT Min'
  'Fwd PSH Flags'
  'Bwd PSH Flags'
  'Fwd URG Flags'
  'Bwd URG Flags'
  'Fwd Header Len'
  'Bwd Header Len'
  'Fwd Pkts/s'
  'Bwd Pkts/s'
  'Pkt Len Min'
  'Pkt Len Max'
  'Pkt Len Mean'
  'Pkt Len Std'
  'Pkt Len Var'
  'FIN Flag Cnt'
  'S

In [3]:
X_train = pd.read_csv(os.path.join(data_2017_path, "X_train.csv"))
original_features = set(X_train.columns)
new_features = set(df_2018_sample.columns)

print(f"2017 features: {len(original_features)}")
print(f"2018 features: {len(new_features)}")

common = original_features & new_features
print(f"\nCommon features: {len(common)}")
print(sorted(common))

only_2017 = original_features - new_features
only_2018 = new_features - original_features

print(f"\nOnly in 2017 ({len(only_2017)}):")
print(only_2017)

print(f"\nOnly in 2018 ({len(only_2018)}):")
print(only_2018)

2017 features: 80
2018 features: 80

Common features: 28
['Active Max', 'Active Mean', 'Active Min', 'Active Std', 'Bwd IAT Max', 'Bwd IAT Mean', 'Bwd IAT Min', 'Bwd IAT Std', 'Bwd PSH Flags', 'Bwd URG Flags', 'CWE Flag Count', 'Down/Up Ratio', 'Flow Duration', 'Flow IAT Max', 'Flow IAT Mean', 'Flow IAT Min', 'Flow IAT Std', 'Fwd IAT Max', 'Fwd IAT Mean', 'Fwd IAT Min', 'Fwd IAT Std', 'Fwd PSH Flags', 'Fwd URG Flags', 'Idle Max', 'Idle Mean', 'Idle Min', 'Idle Std', 'Protocol']

Only in 2017 (52):
{'Bwd Packet Length Std', 'Subflow Fwd Packets', 'Min Packet Length', 'min_seg_size_forward', 'Packet Length Variance', 'Max Packet Length', 'FIN Flag Count', 'Fwd Packet Length Max', 'Fwd Packet Length Std', 'Init_Win_bytes_forward', 'Bwd IAT Total', 'Bwd Packets/s', 'Total Length of Fwd Packets', 'Avg Bwd Segment Size', 'Fwd Header Length.1', 'act_data_pkt_fwd', 'Packet Length Mean', 'Avg Fwd Segment Size', 'Subflow Fwd Bytes', 'Bwd Header Length', 'Packet Length Std', 'Destination Port', '

In [4]:
label_col_2018 = [c for c in df_2018_sample.columns if 'label' in c.lower()][0]
print(f"Label column: '{label_col_2018}'")
print(f"\nLabel values:")
print(df_2018_sample[label_col_2018].value_counts())

Label column: 'Label'

Label values:
Label
DoS attacks-GoldenEye    4947
Benign                     53
Name: count, dtype: int64


In [5]:
column_mapping = {
    'Dst Port': 'Destination Port',
    'Tot Fwd Pkts': 'Total Fwd Packets',
    'Tot Bwd Pkts': 'Total Backward Packets',
    'TotLen Fwd Pkts': 'Total Length of Fwd Packets',
    'TotLen Bwd Pkts': 'Total Length of Bwd Packets',
    'Fwd Pkt Len Max': 'Fwd Packet Length Max',
    'Fwd Pkt Len Min': 'Fwd Packet Length Min',
    'Fwd Pkt Len Mean': 'Fwd Packet Length Mean',
    'Fwd Pkt Len Std': 'Fwd Packet Length Std',
    'Bwd Pkt Len Max': 'Bwd Packet Length Max',
    'Bwd Pkt Len Min': 'Bwd Packet Length Min',
    'Bwd Pkt Len Mean': 'Bwd Packet Length Mean',
    'Bwd Pkt Len Std': 'Bwd Packet Length Std',
    'Flow Byts/s': 'Flow Bytes/s',
    'Flow Pkts/s': 'Flow Packets/s',
    'Fwd IAT Tot': 'Fwd IAT Total',
    'Bwd IAT Tot': 'Bwd IAT Total',
    'Fwd Header Len': 'Fwd Header Length',
    'Bwd Header Len': 'Bwd Header Length',
    'Fwd Pkts/s': 'Fwd Packets/s',
    'Bwd Pkts/s': 'Bwd Packets/s',
    'Pkt Len Min': 'Min Packet Length',
    'Pkt Len Max': 'Max Packet Length',
    'Pkt Len Mean': 'Packet Length Mean',
    'Pkt Len Std': 'Packet Length Std',
    'Pkt Len Var': 'Packet Length Variance',
    'FIN Flag Cnt': 'FIN Flag Count',
    'SYN Flag Cnt': 'SYN Flag Count',
    'RST Flag Cnt': 'RST Flag Count',
    'PSH Flag Cnt': 'PSH Flag Count',
    'ACK Flag Cnt': 'ACK Flag Count',
    'URG Flag Cnt': 'URG Flag Count',
    'ECE Flag Cnt': 'ECE Flag Count',
    'Pkt Size Avg': 'Average Packet Size',
    'Fwd Seg Size Avg': 'Avg Fwd Segment Size',
    'Bwd Seg Size Avg': 'Avg Bwd Segment Size',
    'Fwd Byts/b Avg': 'Fwd Avg Bytes/Bulk',
    'Fwd Pkts/b Avg': 'Fwd Avg Packets/Bulk',
    'Fwd Blk Rate Avg': 'Fwd Avg Bulk Rate',
    'Bwd Byts/b Avg': 'Bwd Avg Bytes/Bulk',
    'Bwd Pkts/b Avg': 'Bwd Avg Packets/Bulk',
    'Bwd Blk Rate Avg': 'Bwd Avg Bulk Rate',
    'Subflow Fwd Pkts': 'Subflow Fwd Packets',
    'Subflow Fwd Byts': 'Subflow Fwd Bytes',
    'Subflow Bwd Pkts': 'Subflow Bwd Packets',
    'Subflow Bwd Byts': 'Subflow Bwd Bytes',
    'Init Fwd Win Byts': 'Init_Win_bytes_forward',
    'Init Bwd Win Byts': 'Init_Win_bytes_backward',
    'Fwd Act Data Pkts': 'act_data_pkt_fwd',
    'Fwd Seg Size Min': 'min_seg_size_forward',
}

df_2018_sample.rename(columns=column_mapping, inplace=True)

# Re-check overlap after renaming
new_features_aligned = set(df_2018_sample.columns)
common_aligned = original_features & new_features_aligned

print(f"Common features after alignment: {len(common_aligned)}")
print(f"\nStill missing from 2017 perspective:")
print(original_features - new_features_aligned)

Common features after alignment: 78

Still missing from 2017 perspective:
{'Fwd Header Length.1', 'Source Port'}


In [6]:
def load_and_align_2018(file_path, column_mapping):
    df = pd.read_csv(file_path, low_memory=False)
    df.columns = df.columns.str.strip()
    df.rename(columns=column_mapping, inplace=True)
    return df

# Load all 3 files fully (not just samples)
dfs_2018 = []
for file in csv_files_2018:
    path = os.path.join(data_2018_path, file)
    print(f"Loading {file}...")
    df_temp = load_and_align_2018(path, column_mapping)
    print(f"  -> {df_temp.shape}")
    dfs_2018.append(df_temp)

df_2018_full = pd.concat(dfs_2018, ignore_index=True)
print(f"\nCombined 2018 shape: {df_2018_full.shape}")
print(f"\nLabel distribution:")
print(df_2018_full['Label'].value_counts())

Loading Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv...
  -> (1048575, 80)
Loading Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv...
  -> (1048575, 80)
Loading Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv...
  -> (1048575, 80)

Combined 2018 shape: (3145725, 80)

Label distribution:
Label
Benign                   2405123
DDOS attack-HOIC          686012
DoS attacks-GoldenEye      41508
DoS attacks-Slowloris      10990
DDOS attack-LOIC-UDP        1730
Brute Force -Web             249
Brute Force -XSS              79
SQL Injection                 34
Name: count, dtype: int64


In [8]:
# Drop Timestamp (not used as a feature, same as 2017 preprocessing)
df_2018_full.drop(columns=['Timestamp'], inplace=True, errors='ignore')

# Encode binary label
df_2018_full['Label_Binary'] = df_2018_full['Label'].apply(
    lambda x: 0 if x.strip().lower() == 'benign' else 1
)

# Handle inf and NaN
numeric_cols = df_2018_full.select_dtypes(include=[np.number]).columns
df_2018_full[numeric_cols] = df_2018_full[numeric_cols].replace([np.inf, -np.inf], np.nan)
df_2018_full[numeric_cols] = df_2018_full[numeric_cols].fillna(df_2018_full[numeric_cols].median())

print(f"Shape after cleaning: {df_2018_full.shape}")
print(f"Missing values: {df_2018_full.isnull().sum().sum()}")
print(f"\nBinary label distribution:")
print(df_2018_full['Label_Binary'].value_counts())

Shape after cleaning: (3145725, 80)
Missing values: 0

Binary label distribution:
Label_Binary
0    2405123
1     740602
Name: count, dtype: int64


In [9]:
common_features = sorted(list(common_aligned))
print(f"Using {len(common_features)} common features")
print(common_features)

Using 78 common features
['ACK Flag Count', 'Active Max', 'Active Mean', 'Active Min', 'Active Std', 'Average Packet Size', 'Avg Bwd Segment Size', 'Avg Fwd Segment Size', 'Bwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Header Length', 'Bwd IAT Max', 'Bwd IAT Mean', 'Bwd IAT Min', 'Bwd IAT Std', 'Bwd IAT Total', 'Bwd PSH Flags', 'Bwd Packet Length Max', 'Bwd Packet Length Mean', 'Bwd Packet Length Min', 'Bwd Packet Length Std', 'Bwd Packets/s', 'Bwd URG Flags', 'CWE Flag Count', 'Destination Port', 'Down/Up Ratio', 'ECE Flag Count', 'FIN Flag Count', 'Flow Bytes/s', 'Flow Duration', 'Flow IAT Max', 'Flow IAT Mean', 'Flow IAT Min', 'Flow IAT Std', 'Flow Packets/s', 'Fwd Avg Bulk Rate', 'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Header Length', 'Fwd IAT Max', 'Fwd IAT Mean', 'Fwd IAT Min', 'Fwd IAT Std', 'Fwd IAT Total', 'Fwd PSH Flags', 'Fwd Packet Length Max', 'Fwd Packet Length Mean', 'Fwd Packet Length Min', 'Fwd Packet Length Std', 'Fwd Packets/s', 'F

In [10]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import time

# Reload original 2017 training data
X_train_full = pd.read_csv(os.path.join(data_2017_path, "X_train.csv"))
y_train_full = pd.read_csv(os.path.join(data_2017_path, "y_train.csv")).squeeze()

X_train_common = X_train_full[common_features]

print(f"Retraining on {X_train_common.shape[1]} common features...")
start = time.time()

rf_cross = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    n_jobs=-1,
    random_state=42,
    verbose=1
)
rf_cross.fit(X_train_common, y_train_full)

print(f"Training complete in {time.time() - start:.1f} seconds")

Retraining on 78 common features...


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=-1)]: Done  42 tasks      | elapsed:  1.8min


Training complete in 239.1 seconds


[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:  4.0min finished


In [11]:
X_test_2017 = pd.read_csv(os.path.join(data_2017_path, "X_test.csv"))[common_features]
y_test_2017 = pd.read_csv(os.path.join(data_2017_path, "y_test.csv")).squeeze()

preds_2017 = rf_cross.predict(X_test_2017)
print(f"Accuracy on 2017 test set (common features only): {accuracy_score(y_test_2017, preds_2017):.4f}")

[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.7s


Accuracy on 2017 test set (common features only): 0.9991


[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    1.7s finished


In [12]:
X_test_2018 = df_2018_full[common_features]
y_test_2018 = df_2018_full['Label_Binary']

preds_2018 = rf_cross.predict(X_test_2018)

print("=" * 55)
print("   CROSS-DATASET VALIDATION — CICIDS2018")
print("=" * 55)
print(classification_report(y_test_2018, preds_2018, target_names=["BENIGN", "ATTACK"]))
print(f"Accuracy: {accuracy_score(y_test_2018, preds_2018):.4f}")

[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    3.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    7.4s finished


   CROSS-DATASET VALIDATION — CICIDS2018


C:\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


              precision    recall  f1-score   support

      BENIGN       0.76      1.00      0.87   2405123
      ATTACK       0.00      0.00      0.00    740602

    accuracy                           0.76   3145725
   macro avg       0.38      0.50      0.43   3145725
weighted avg       0.58      0.76      0.66   3145725

Accuracy: 0.7646


C:\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [13]:
import sys
sys.path.append(r"C:\Users\lenovo\Desktop\Projects\Network-IDS\src")
from preprocess import load_and_sample, encode_labels, DATA_PATH

# This reloads and recreates the raw combined dataframe (same as Phase 3)
print("Reloading raw CICIDS2017 data...")
df_raw_2017 = load_and_sample(DATA_PATH, benign_frac=0.3)
df_raw_2017, le_temp = encode_labels(df_raw_2017)

print(f"\nRaw combined shape: {df_raw_2017.shape}")

Reloading raw CICIDS2017 data...
Loading Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv...
  -> (157342, 81) | Attacks: 128027
Loading Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv...
  -> (197191, 81) | Attacks: 158930
Loading Friday-WorkingHours-Morning.pcap_ISCX.csv...
  -> (58686, 81) | Attacks: 1966
Loading Monday-WorkingHours.pcap_ISCX.csv...
  -> (158975, 81) | Attacks: 0
Loading Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv...
  -> (86606, 81) | Attacks: 36
Loading Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv...
  -> (341238, 81) | Attacks: 290782
Loading Tuesday-WorkingHours.pcap_ISCX.csv...
  -> (143457, 81) | Attacks: 13835
Loading Wednesday-workingHours.pcap_ISCX.csv...
  -> (384681, 81) | Attacks: 252672

Combined shape: (1528176, 81)

Raw combined shape: (1528176, 83)


In [14]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

X_raw = df_raw_2017[common_features]
y_raw = df_raw_2017['Label_Binary']

X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
    X_raw, y_raw, test_size=0.2, random_state=42, stratify=y_raw
)

# Fit scaler ONLY on training data
scaler_common = StandardScaler()
X_train_scaled = pd.DataFrame(scaler_common.fit_transform(X_train_raw), columns=common_features)
X_test_scaled  = pd.DataFrame(scaler_common.transform(X_test_raw),     columns=common_features)

print(f"X_train_scaled: {X_train_scaled.shape}")
print(f"X_test_scaled:  {X_test_scaled.shape}")

X_train_scaled: (1222540, 78)
X_test_scaled:  (305636, 78)


In [15]:
rf_cross_fixed = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    n_jobs=-1,
    random_state=42,
    verbose=1
)
rf_cross_fixed.fit(X_train_scaled, y_train_raw)

preds_2017_fixed = rf_cross_fixed.predict(X_test_scaled)
print(f"2017 sanity check accuracy: {accuracy_score(y_test_raw, preds_2017_fixed):.4f}")

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=-1)]: Done  42 tasks      | elapsed:  1.6min
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:  3.7min finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.7s


2017 sanity check accuracy: 0.9991


[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    1.7s finished


In [16]:
X_test_2018_scaled = pd.DataFrame(
    scaler_common.transform(df_2018_full[common_features]),
    columns=common_features
)
y_test_2018 = df_2018_full['Label_Binary']

preds_2018_fixed = rf_cross_fixed.predict(X_test_2018_scaled)

print("=" * 55)
print("   CROSS-DATASET VALIDATION — CICIDS2018 (FIXED)")
print("=" * 55)
print(classification_report(y_test_2018, preds_2018_fixed, target_names=["BENIGN", "ATTACK"]))
print(f"Accuracy: {accuracy_score(y_test_2018, preds_2018_fixed):.4f}")

[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    6.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:   13.3s finished


   CROSS-DATASET VALIDATION — CICIDS2018 (FIXED)
              precision    recall  f1-score   support

      BENIGN       0.77      1.00      0.87   2405123
      ATTACK       0.98      0.03      0.05    740602

    accuracy                           0.77   3145725
   macro avg       0.88      0.51      0.46   3145725
weighted avg       0.82      0.77      0.68   3145725

Accuracy: 0.7708
